# Contrast, Histograms, and Image Enhancement

> **Beginner · Image processing**


## Why this matters

A histogram explains why an image looks flat, clipped, or poorly exposed. Enhancement should improve information for a task, not just create a dramatic image.

**Where it appears:** Low-light preprocessing, exposure normalization, image similarity checks, and visualization.


## Learning Objectives

- Adjust brightness/contrast correctly using linear transforms
- Apply gamma correction and understand why it differs from linear brightness
- Sharpen images using unsharp masking
- Compute and interpret grayscale and color histograms
- Apply global equalization and CLAHE, and know when each is appropriate
- Use histogram comparison for simple image-similarity measurement


## Prerequisites

07 Filtering, Convolution, and Noise

Work through the examples in order. Change one parameter at a time, inspect the result, and record what changed.


## Core OpenCV APIs

`cv2.convertScaleAbs`, LUTs, `cv2.calcHist`, `cv2.equalizeHist`, `cv2.createCLAHE`

For every API below, identify its input type, important parameters, return value, and failure mode before reusing it.


## Conceptual Foundation


### Image Enhancement and Contrast

Brightness/contrast adjustment is the linear transform `output = alpha *
input + beta` (alpha controls contrast, beta controls brightness).
**Gamma correction** (`output = 255 * (input/255)^gamma`) is a *non-linear*
adjustment matching human perceptual sensitivity, correcting images that
look uniformly too dark/bright in the mid-tones without clipping highlights
the way a linear boost would. **Unsharp masking** sharpens by subtracting
a blurred version of the image from itself, amplifying high-frequency
(edge) detail.


### Histograms and Histogram Equalization

A histogram counts pixel intensities. Global **histogram equalization**
redistributes intensities to use the full 0-255 range, improving contrast --
but it operates globally, which can over-amplify noise in already-noisy
regions and under-improve local areas. **CLAHE** (Contrast-Limited Adaptive
Histogram Equalization) equalizes locally in tiles with a clip limit to
prevent noise amplification -- almost always the better choice for
real photographs.


## Setup

Run this cell once. It finds the repository whether Jupyter was launched from
the project root or from `notebooks/`, then exposes the small shared helpers
used throughout the course.


In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")

_candidates = (Path.cwd(), Path.cwd().parent)
REPO_ROOT = next(
    (path for path in _candidates if (path / "utils" / "cv_utils.py").exists()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from the repository root or notebooks/ directory.")

UTILS_DIR = REPO_ROOT / "utils"
if str(UTILS_DIR) not in sys.path:
    sys.path.insert(0, str(UTILS_DIR))

import cv2
import matplotlib.pyplot as plt
import numpy as np

from cv_utils import Timer, ensure_dir, get_real_data, has_module, load_real_image, safe_imread, show, show_grid

print("OpenCV:", cv2.__version__)
print("Repository:", REPO_ROOT)


## Guided Lessons


## Part 1: Image Enhancement and Contrast


### 1. Linear brightness/contrast

Implement the alpha/beta transform explicitly, with clipping, and show contrast vs brightness are independent controls.


In [ ]:
import cv2
import numpy as np
from cv_utils import load_real_image, get_real_data, show_grid


def adjust_contrast_brightness(
    image: np.ndarray, alpha: float = 1.0, beta: int = 0
) -> np.ndarray:
    """alpha > 1 increases contrast, beta > 0 increases brightness. Both clipped to valid range."""
    return np.clip(image.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)


scene = load_real_image("images/landscapes", "city.jpg")
more_contrast = adjust_contrast_brightness(scene, alpha=1.6, beta=0)
brighter = adjust_contrast_brightness(scene, alpha=1.0, beta=60)

show_grid(
    [
        ("original", scene),
        ("alpha=1.6 (more contrast)", more_contrast),
        ("beta=+60 (brighter)", brighter),
    ]
)

### 2. Gamma correction

Compare linear brightness boost to gamma correction on a dark image -- gamma lifts shadows/midtones without blowing out the highlights the way linear beta does.


In [ ]:
def adjust_gamma(image: np.ndarray, gamma: float) -> np.ndarray:
    """gamma < 1 brightens midtones/shadows more than highlights; gamma > 1 darkens them."""
    inv_gamma = 1.0 / gamma
    table = np.array(
        [(i / 255.0) ** inv_gamma * 255 for i in range(256)], dtype=np.uint8
    )
    return cv2.LUT(image, table)


dark_scene = adjust_contrast_brightness(scene, alpha=0.5, beta=0)
linear_fix = adjust_contrast_brightness(dark_scene, alpha=1.0, beta=80)
gamma_fix = adjust_gamma(dark_scene, gamma=0.5)

show_grid(
    [
        ("darkened", dark_scene),
        ("linear brighten (beta)", linear_fix),
        ("gamma correction", gamma_fix),
    ]
)

### 3. Sharpening with unsharp masking

Blur the image, subtract the blur from the original to isolate high-frequency detail, then add that detail back with a weight -- the standard sharpening recipe.


In [ ]:
def unsharp_mask(
    image: np.ndarray, amount: float = 1.0, blur_ksize: int = 9
) -> np.ndarray:
    blurred = cv2.GaussianBlur(image, (blur_ksize, blur_ksize), 0)
    sharpened = cv2.addWeighted(image, 1 + amount, blurred, -amount, 0)
    return sharpened


sharpened = unsharp_mask(scene, amount=1.2)
show_grid([("original", scene), ("unsharp masked", sharpened)])

## Part 2: Histograms and Histogram Equalization


### 1. Computing and reading histograms

`cv2.calcHist` produces the raw counts; visualize alongside the source image to make the connection between pixel distribution and visual contrast explicit.


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from cv_utils import load_real_image, get_real_data


def plot_gray_histogram(gray: np.ndarray, title: str) -> None:
    hist = cv2.calcHist([gray], [0], None, [256], [0, 256])
    plt.figure(figsize=(5, 3))
    plt.plot(hist)
    plt.title(title)
    plt.xlabel("intensity")
    plt.ylabel("pixel count")
    plt.tight_layout()
    plt.show()


low_contrast = (
    load_real_image("images/standard", "dog.jpg", cv2.IMREAD_GRAYSCALE).astype(
        np.float32
    )
    * 0.35
    + 90
).astype(np.uint8)

plot_gray_histogram(
    low_contrast, "Low-contrast image: histogram bunched in a narrow range"
)

### 2. Global equalization and its risk

`cv2.equalizeHist` fixes the low-contrast case well, but amplifies noise when the image already has noise -- shown directly by comparing equalized noisy vs clean images.


In [ ]:
from cv_utils import load_real_image, get_real_data, show_grid

equalized = cv2.equalizeHist(low_contrast)

noise = np.random.normal(0, 8, low_contrast.shape)
noisy_low_contrast = np.clip(low_contrast.astype(np.float32) + noise, 0, 255).astype(
    np.uint8
)
equalized_noisy = cv2.equalizeHist(noisy_low_contrast)

show_grid(
    [
        ("low contrast", low_contrast),
        ("equalized (clean input)", equalized),
        ("low contrast + noise", noisy_low_contrast),
        ("equalized (noise amplified!)", equalized_noisy),
    ]
)

### 3. CLAHE: the safer default

CLAHE's tile-based, clip-limited approach improves local contrast without the same noise-amplification risk -- compare directly against global equalization on the same noisy input.


In [ ]:
def apply_clahe(
    gray: np.ndarray, clip_limit: float = 2.0, tile_grid=(8, 8)
) -> np.ndarray:
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    return clahe.apply(gray)


clahe_result = apply_clahe(noisy_low_contrast)
show_grid(
    [("global equalize (noisy)", equalized_noisy), ("CLAHE (noisy)", clahe_result)]
)

### 4. Histogram comparison for similarity

Compare two images' color histograms using correlation -- a fast, simple similarity signal useful for tasks like duplicate/near-duplicate detection.


In [ ]:
def color_histogram(image: np.ndarray, bins: int = 32) -> np.ndarray:
    hist = cv2.calcHist([image], [0, 1, 2], None, [bins] * 3, [0, 256] * 3)
    return cv2.normalize(hist, hist).flatten()


scene_a = load_real_image("images/objects", "coins.jpg")
noise_b = np.random.normal(0, 10, scene_a.shape)
scene_b = np.clip(scene_a.astype(np.float32) + noise_b, 0, 255).astype(
    np.uint8
)  # near-duplicate

# OpenCV loads images as 3-channels (BGR) by default, so scene_c is already 3 channels.
scene_c = load_real_image("images/documents", "text.png")  # unrelated

h_a, h_b, h_c = (
    color_histogram(scene_a),
    color_histogram(scene_b),
    color_histogram(scene_c),  # Pass scene_c directly
)

sim_ab = cv2.compareHist(h_a, h_b, cv2.HISTCMP_CORREL)
sim_ac = cv2.compareHist(h_a, h_c, cv2.HISTCMP_CORREL)
print(f"similarity(scene, noisy scene)   = {sim_ab:.3f}  (expect high)")
print(f"similarity(scene, checkerboard)  = {sim_ac:.3f}  (expect low)")

## Mini Projects

Complete one project unaided before reading any provided solution. Extend it with a parameter, dataset, or failure case of your own.


### Mini Project — Image Enhancement and Contrast: Local Adaptive Gamma Correction

Standard global gamma correction applies the same enhancement function uniformly, which often oversaturates bright regions while attempting to reveal shadow details. Here, we build an adaptive correction algorithm that estimates localized illumination to enhance high-contrast images.


In [ ]:
# Generate a high-contrast synthetic image (left side dark, right side bright)
img = load_real_image("images/landscapes", "city.jpg")
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
mask = np.tile(np.linspace(0.1, 1.0, img.shape[1]), (img.shape[0], 1))
uneven_lit = np.clip(gray * mask, 0, 255).astype(np.uint8)

# Estimate local illumination using large Gaussian blur
illumination = cv2.GaussianBlur(uneven_lit, (51, 51), 0)

# Calculate localized adaptive gamma: lower gamma in dark areas, higher in bright areas
gamma_map = np.clip(illumination.astype(np.float32) / 255.0, 0.1, 1.0)

# Apply element-wise adaptive gamma correction
norm_img = uneven_lit.astype(np.float32) / 255.0
corrected = np.power(norm_img, gamma_map)
corrected_uint8 = (corrected * 255.0).astype(np.uint8)

print("Local adaptive gamma correction complete.")
show_grid(
    [
        ("Shadowed Scene", uneven_lit),
        ("Local Illumination Estimate", illumination),
        ("Adaptive Correction", corrected_uint8),
    ]
)

### Mini Project — Histograms and Histogram Equalization: Object Segmentation via Histogram Backprojection

Histogram backprojection matches pixels in a target scene with a color distribution histogram of a region of interest (the query object). This is widely used in tracking applications to locate colored targets.


In [ ]:
img = load_real_image("images/objects", "coins.jpg")
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

# Define query object: crop the green circular shape
roi = hsv[40:160, 270:390]

# Calculate normalized Hue-Saturation histogram for the target object
roi_hist = cv2.calcHist([roi], [0, 1], None, [180, 256], [0, 180, 0, 256])
cv2.normalize(roi_hist, roi_hist, 0, 255, cv2.NORM_MINMAX)

# Calculate backprojection map
backproj = cv2.calcBackProject([hsv], [0, 1], roi_hist, [0, 180, 0, 256], 1)

# Apply threshold to binarize matching regions
_, thresh = cv2.threshold(backproj, 50, 255, cv2.THRESH_BINARY)

print("Histogram backprojection complete.")
show_grid(
    [
        ("Target Scene", img),
        ("Backproject Probability", backproj),
        ("Threshold Segmented Target", thresh),
    ]
)

## Exercises

Attempt the beginner, intermediate, and advanced prompts in order. Keep notes on assumptions and failures, not just successful output.


### Exercises — Image Enhancement and Contrast
1. Build `auto_brightness(image, target_mean=128)` that computes and applies the beta needed to hit a target mean intensity.
2. Plot gamma-corrected output curves for gamma in [0.4, 0.7, 1.0, 1.5, 2.5] on one matplotlib chart.
3. Sweep unsharp masking's `amount` from 0.5 to 3.0 and note when halo artifacts start appearing at edges.

Use the empty cell below to work through them.


#### Solutions — Image Enhancement and Contrast

In [ ]:
# Solution 1: auto_brightness
def auto_brightness(image: np.ndarray, target_mean: float = 128.0) -> np.ndarray:
    """Adjust BGR image brightness (beta offset) to match a target mean value."""
    current_mean = np.mean(image)
    beta = target_mean - current_mean
    return np.clip(image.astype(np.float32) + beta, 0, 255).astype(np.uint8)

In [ ]:
# Solution 2: Plot gamma-corrected curves on one chart
def plot_gamma_curves() -> None:
    """Plot transfer curves for multiple gamma settings."""
    x = np.linspace(0, 255, 256)
    gammas = [0.4, 0.7, 1.0, 1.5, 2.5]

    plt.figure(figsize=(6, 4))
    for g in gammas:
        y = np.clip(np.power(x / 255.0, g) * 255.0, 0, 255)
        plt.plot(x, y, label=f"Gamma {g}")
    plt.title("Gamma Correction Curves")
    plt.xlabel("Input Value")
    plt.ylabel("Output Value")
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
# Solution 3: Sweep unsharp masking amount parameter
# Explanation: The unsharp masking parameter `amount` controls the scaling factor of high-pass
# details added back to the original image. Lower values (0.5) produce subtle sharpening.
# Increasing values above 1.5 start producing harsh halo artifacts (bright or dark ring boundaries)
# along sharp edges and amplify sensor noise.

### Exercises — Histograms and Histogram Equalization
1. Plot histograms for each BGR channel separately (3 overlaid line plots) for a color scene.
2. Sweep CLAHE's `clip_limit` (1, 2, 4, 8) and describe when it starts over-amplifying noise too.
3. Implement `most_similar(query_hist, candidate_hists)` returning the index of the best `HISTCMP_CORREL` match.

Use the empty cell below to work through them.



#### Solutions — Histograms and Histogram Equalization

In [ ]:
# Solution 1: Plot histograms for each BGR channel separately
def plot_bgr_histograms(image: np.ndarray) -> None:
    """Display separate BGR color histogram curves on one graph."""
    colors = ("b", "g", "r")
    plt.figure(figsize=(6, 4))
    for i, col in enumerate(colors):
        hist = cv2.calcHist([image], [i], None, [256], [0, 256])
        plt.plot(hist, color=col, label=f"Channel {i}")
    plt.title("BGR Color Histograms")
    plt.xlabel("Pixel Intensity")
    plt.ylabel("Count")
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
# Solution 2: Sweep CLAHE clip_limit
# Explanation: `clipLimit` restricts the maximum height of local histogram bins.
# A value of 1.0 means no contrast amplification (equal to original). Values of 2.0-4.0
# provide standard contrast enhancement. Setting it to 8.0+ leads to extreme, unnatural contrast
# changes, which over-amplifies camera sensor noise in flat background areas.

In [ ]:
# Solution 3: most_similar histogram matcher
def most_similar(query_hist: np.ndarray, candidate_hists: list[np.ndarray]) -> int:
    """Return the index of the candidate histogram that best matches query_hist using correlation."""
    best_score = -1.0
    best_idx = -1

    for i, cand in enumerate(candidate_hists):
        score = cv2.compareHist(query_hist, cand, cv2.HISTCMP_CORREL)
        if score > best_score:
            best_score = score
            best_idx = i

    return best_idx


# Test functions
img = load_real_image("images/objects", "coins.jpg")
plot_bgr_histograms(img)

## Summary

You can diagnose tone and contrast with histograms, choose global or local enhancement, and quantify image similarity where appropriate.

- **Best Practices:** Work in a suitable color space, preserve originals, avoid clipping, and inspect local artifacts after equalization or sharpening.
- **Common Pitfalls:** Equalizing color channels independently, treating histograms as spatial information, and over-sharpening noisy inputs.